# PokéLLM
This notebook contains an example LLM agent built using `LangChain` for querying technical details about the main series Pokémon video games.

## 1. Setup
Install and import relevant libraries, and define all *tool* functions.

In [92]:
%%capture
%pip install langchain langchain_openai langchain_community lancedb

In [93]:
import numpy as np
import requests
from typing import Tuple
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor
from langchain_classic import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import LanceDB
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os; os.environ['OPENAI_API_KEY'] = ### PUT YOUR API KEY HERE ###

### Tools

In [94]:
def find_start_tag(text:str, tag:str, after:int=None, before:int=None) -> int:
    """
    To account for tags having parameters, this function searches for both cases.
    """
    start_1 = text.find(f'<{tag}>', after, before)
    start_2 = text.find(f'<{tag} ', after, before)
    if start_1 == -1:
        return start_2
    elif start_2 == -1:
        return start_1
    return min([start_1, start_2])

def find_tag(text:str, tag:str, after_index:int=0, before_index:int=None) -> Tuple[str, int]:
    """
    Extracts all content within a specified tag.

    Args:
        text (str): The provided text to search through.
        tag (str): The provided tag to search for.
        after_index (int): An index after which the search should begin.

    Returns:
        str: The extracted text.
        int: The index of the end of the end tag.
    """
    # Find initial start and end tags
    start = find_start_tag(text, tag, after_index, before_index)
    end = text.find(f'</{tag}>', start)

    # Break and return if no start tag is found
    if start == -1:
        return '', -1

    # Special treatment for p tags with no closing tag
    if tag == 'p' and '<p>' in text[start:end]:
        end_of_start = text.find('>', start) + 1
        return text[end_of_start:text.find('.', start)+1], text.find('.', start)+1

    # To consider the case where the same tags are embedded within the main one, iteratively search for more
    sub_start = find_start_tag(text, tag, start+1, end)
    while sub_start > -1:
        # Find the next end tag if sub-tags are found
        end = text.find(f'</{tag}>', end+1)
        sub_start = find_start_tag(text, tag, sub_start+1, end)

    end_of_start = text.find('>', start) + 1
    return text[end_of_start:end], end

def strip_tags(text:str, tags:list[str]) -> str:
    """
    This function strips out the start and end tags from some text using a specified list of tags.

    Args:
        text (str): The text to remove tags from.
        tags: (list[str]): A list of tags to be removed.

    Returns:
        str: The stripped text.
    """
    text_stripped = text
    for tag in tags:
        # For each tag, find up to 1000 instances and remove them
        I = 0
        while I < 1000:
            # Finding the start tag
            start_1 = find_start_tag(text_stripped, tag)
            start_2 = text_stripped.find('>', start_1)
            if start_1 >= 0:
                # If start tag found, remove
                text_stripped = text_stripped[:start_1] + (text_stripped[start_2+1:] if start_2 > -1 else '')

            # Finding the end tag
            end_1 = text_stripped.find(f'</{tag}>')
            end_2 = text_stripped.find('>', end_1)
            if end_1 >= 0:
                # If end tag found, remove
                text_stripped = text_stripped[:end_1] + (text_stripped[end_2+1:] if end_2 > -1 else '')

            # If no more tags are being found, then break
            if start_1 < 0 and end_1 < 0:
                break

            I += 1

    return text_stripped

def info_type_evos(name:str) -> str:
    """
    This function can be used to lookup some facts about a specified Pokémon including it's type, Pokédex entries, and it's evolutions.

    Args:
        name (str): The name of the Pokémon to look up.

    Returns:
        str: A concise text summary about the Pokémon.
    """
    # Making HTTPS request
    result = requests.get(f'https://pokemondb.net/pokedex/{name.lower()}')
    html = result.text

    # If successful request, extract information
    if result.ok:
        # The top of the page includes a few useful statements
        paragraphs = []
        I, i = 0, 0
        while I < 5:
            para, i = find_tag(html, 'p', i)
            if para.startswith('<a rel="lightbox"'):
                # Break when you get to the picture
                break
            paragraphs.append(strip_tags(para, ['em', 'a', 'ab', 'q']))
            I += 1

        text_summary = '\n'.join(paragraphs)

        # Gathering key facts from the first table
        facts = strip_tags(find_tag(html, 'table', html.find('<h2>Pokédex data</h2>'))[0], ['tbody', 'tr', 'th', 'td', 'strong', 'a', 'span', 'small'])
        facts = facts.replace('<br>', ' ').replace('<\br>', ' ').replace('\t', '').replace('\n\n\n', '\n\n')

        # Gathering some training facts
        training_facts = strip_tags(find_tag(html, 'table', html.find('<h2>Training</h2>'))[0], ['tbody', 'tr', 'th', 'td', 'a', 'small'])
        training_facts = training_facts.replace('\t', '').replace('\n\n\n', '\n\n')

        # Gathering some breeding facts
        breeding_facts = strip_tags(find_tag(html, 'table', html.find('<h2>Breeding</h2>'))[0], ['tbody', 'tr', 'th', 'td', 'a', 'span', 'small'])
        breeding_facts = breeding_facts.replace('\t', '').replace('\n\n\n', '\n\n')

        # Gathering the evolution information
        evo_info, i = find_tag(html, 'div', html.find('<h2>Evolution'))
        while html.find('infocard-list-evo', i) > -1:
            evo_info_, i = find_tag(html, 'div', i)
            evo_info += evo_info_
        evo_info = strip_tags(evo_info, ['div', 'span', 'a', 'picture', 'source', 'img', 'small', 'i', 'br'])
        evo_info = evo_info.replace(' &middot;', '')

        return text_summary + facts + training_facts + breeding_facts + evo_info
    else:
        raise Exception('Invalid Pokémon name provided')

def stats(name:str) -> str:
    """
    This function can be used to lookup the base stats a specified Pokémon.

    Args:
        name (str): The name of the Pokémon to look up.

    Returns:
        str: A concise text summary of their stats.
    """
    # Making HTTPS request
    result = requests.get(f'https://pokemondb.net/pokedex/{name.lower()}')
    html = result.text

    # If successful request, extract information
    if result.ok:
        # Gathering the base stats
        base_stats = strip_tags(find_tag(html, 'table', html.find('<h2>Base stats</h2>'))[0], ['tbody', 'tr', 'th', 'td', 'div', 'tfoot'])
        base_stats = base_stats.replace('\t', '').replace('\n\n\n', '\n\n').replace('Min', '').replace('Max', '')
        minmax_start = base_stats.find('HP')
        for stat in range(6):
            minmax_start = base_stats.find('\n\n\n', minmax_start)
            base_stats = base_stats[:minmax_start+2] + 'Min Max' + base_stats[minmax_start+2:]

        return base_stats
    else:
        raise Exception('Invalid Pokémon name provided')

def learnset(name:str) -> str:
    """
    This function can be used to lookup the learnset of a specified Pokémon.

    Args:
        name (str): The name of the Pokémon to look up. If you wish to lookup the
                    learnset for a specific generation, then include the generation
                    number directly after the name, separated by a pipe, e.g. "Venonat|3".

    Returns:
        str: A concise text summary of the moves they can learn.
    """
    # Pulling out the specified generation where relevant
    if "|" in name:
        parts = name.split('|')
        name, gen = parts[0], int(parts[1])
    else:
        gen = 9 # Default to the newest generation (currently 9)

    # Making HTTPS request
    result = requests.get(f'https://pokemondb.net/pokedex/{name.lower()}/moves/{gen}')
    html = result.text

    # If successful request, extract information
    if result.ok:

        # String to store all learnset information
        all_moves = ''

        game_st = 0
        G = 0
        while G < 10:
            # Loop through each of the different games in the generation, identified by an ID
            game_st = html.find('#tab-moves-', game_st+10)
            game_ed = html.find('">', game_st)

            G += 1
            if game_st == -1:
                # No more games found
                break

            # Identify the start of the current games information
            moveset_start = html.find(html[game_st+1:game_ed], game_ed)
            game_name = html[html.find('>', game_st)+1:html.find('</a>', game_st)].replace('&#039;', "'")
            game_moves = f'In {game_name}:\n\n'

            for header in ['Moves learnt by level up', 'Egg moves', 'Moves learnt by reminder', 'Moves learnt by TM']:
                # For each type of learnset, find the header to the table
                head = html.find(f'<h3>{header}</h3>', moveset_start)
                if head == -1:
                    continue
                statement = strip_tags(find_tag(html, 'p', head)[0], ['em'])
                if 'does not learn any' in statement or 'cannot be taught' in statement:
                    # If no moves are learnt, continue to the next header
                    continue

                if game_name in ['Legends: Arceus', 'Let\'s Go Pikachu/Let\'s Go Eevee'] and header == 'Egg moves':
                    # Special treatment is needed for Legends: Arceus and Let's Go egg moves
                    continue

                table = strip_tags(find_tag(html, 'table', head)[0], ['thead', 'tr', 'th', 'a', 'div', 'tbody'])
                table = strip_tags(table.replace('</td>', ' '), ['td'])

                start = 0
                while start != -1:
                    # Specific steps to extract the category component from the image
                    start = table.find('title="', start+7)
                    end = table.find('"', start+7)
                    slot = table.find('>', end)+1
                    table = table[:slot] + table[start+7:end] + table[slot:]

                table = strip_tags(table, ['img'])
                redundant_start = table.find('Acc.') + 4
                redundant_end = table.find('Acc.', redundant_start) + 5
                table = table[:redundant_start] + '\n' + table[redundant_end:] # Dealing with some strange formatting

                game_moves += header + '\n' + table + '\n'

            if game_moves != f'In {game_name}:\n\n':
                all_moves += game_moves + '\n'

        if all_moves == '' and gen > 1:
            # If nothing can be found for the current gen, try the generation prior
            return learnset(f'{name}|{gen-1}')
        elif all_moves == '' and gen == 1:
            # Unlikely scenario, but if a Pokémon learns no moves in any gen, they will end up here
            raise Exception('The Pokémon does not learn any moves in the mainline games.')

        return all_moves

    else:
        raise Exception('Invalid Pokémon name or generation specification.')

def move(name:str) -> str:
    """
    This function can be used to lookup details on a specified move.

    Args:
        name (str): The name of the move to lookup.

    Returns:
        str: A concise text summary of details about the move.
    """
    # Making HTTPS request
    result = requests.get(f'https://pokemondb.net/move/{name.lower().replace(' ', '-')}')
    html = result.text

    # If successful request, extract information
    if result.ok:
        # Table with some key facts and figures
        tab = find_tag(html, 'table', html.find('<h2>Move data</h2>'))[0]
        tab = strip_tags(tab, ['tbody', 'th', 'td', 'a', 'img', 'small']).replace('<tr>', '\n')
        tab = strip_tags(tab, ['tr']).replace('&nbsp;', '').replace('\n\t', '\n')

        # Effects description
        desc_start = html.find('>Effects</h2>')
        desc = html[desc_start+13:html.find('<h2', desc_start+13)]
        desc = strip_tags(desc, ['p', 'em', 'a', 'q', 'div', 'b', 'abbr', 'ul', 'li', 'br']).replace('\t', '').\
                replace('\n<h3>Z-Move effects</h3>', '').replace('\n<h3>Changes</h3>', '')

        # Get target information
        targ = find_tag(html, 'p', html.find('<h2>Move target</h2>'))[0]

        return tab.rstrip('\n') + '\n\n' + desc.rstrip('\n') + '\n\n' + targ

def games() -> str:
    """
    This function can be used to get a list of all main series Pokémon games in order of generation.

    Returns:
        str: List of games.
    """
    with open('Games.csv') as file:
        text = file.read()
    return text

def type_adv(types:str) -> str:
    """
    This function can be used to lookup the type advantage/disadvantage between two types.

    Args:
        types (str): The two types in the form [offensive]|[defensive].

    Returns:
        str: Text describing the advantage/disadvantage.
    """
    with open('Type Data.txt') as file:
        text = file.read()
    off, dfn = types.split('|')
    dfn = dfn.replace('-', ' ').replace('/', ' ').replace('\\', ' ')

    if ' ' in dfn:
        dfns = dfn.split(' ')
    else:
        dfns = [dfn]

    out_text = ''
    for dfn in dfns:
        start = text.find(f'{off.title()} against {dfn.title()}')
        if start != -1:
            end = text.find('\n', start)
            out_text += '\n' + text[start:end]
        else:
            return 'Invalid types provided.'

    return out_text

def damage_calc_info(gen:str='') -> str:
    """
    This function can be used to retrieve the list of parameters used to calculate damage in the Pokémon games for a specified generation.

    Args:
        gen (str): The generation to lookup logic for.

    Return:
        str: Retrieved parameters.
    """
    # Making HTTPS request
    result = requests.get('https://bulbapedia.bulbagarden.net/wiki/Damage')
    html = result.text

    # If request successful, extract information
    if result.ok:
        # Converting numerical gens to numerals
        gen_numerals = {'1':'I', '2':'II', '3':'III', '4':'IV'}
        if gen in gen_numerals.keys():
            gen_ = gen_numerals[gen]
        else:
            gen_ = 'V_onward'

        # Specifying the relevant section and getting initial arguments
        sec = html.find(f'id="Generation_{gen_}"')
        args, i = find_tag(html, 'ul', sec)

        # Special treatment is required for gen 5 onwards
        if gen_ == 'V_onward':
            _, j = find_tag(html, 'table', i)
            args += '\n' + find_tag(html, 'dl', j)[0]
            args += '\n' + find_tag(html, 'ul', j)[0]

        # Removing tags, including mathematical ones
        args = strip_tags(args, ['li', 'a', 'ul', 'sup', 'span', 'dd'])
        args = strip_tags(args, ['math', 'mrow', 'mstyle', 'mo', 'mi'])

        # Ensuring mathematical formulas are displayed correctly
        end = 0
        while True:
            start = args.find('<msup><mn>', end)
            if start == -1:
                break
            end = args.find('</msup>', start)
            args = args[:start] + args[start:end].replace('</mn>', '^') + args[end:]

        return strip_tags(args, ['msup', 'mn'])

# Preparing stat calculation info and nature webpages
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
loader = WebBaseLoader('https://bulbapedia.bulbagarden.net/wiki/Stat')
stat_info = loader.load()
loader = WebBaseLoader('https://bulbapedia.bulbagarden.net/wiki/Nature')
nature_info = loader.load()
docs = splitter.split_documents(stat_info + nature_info)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# Removing the previous vector database before creating the new one
%rm -rf ./db
vectorstore = LanceDB.from_documents(docs, embedding=embeddings, uri='./db')
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def stat_calc_info(query:str) -> str:
    """
    This function can be used to retrieve information on how stats are calculated in the Pokémon games.

    Args:
        query (str): The query to lookup with.

    Return:
        str: Retrieved information.
    """
    global retriever
    docs = retriever.invoke(query)
    return "\n\n".join([d.page_content for d in docs])

def damage_calc(params:str) -> int:
    """
    This function can be used to calculate damage in the Pokémon games with specified parameters.

    Args:
        params (str): All parameters, including generation, "gen", provided in a dictionary format as a string literal.

    Returns:
        int: The calculated damage.
    """
    try:
        # Extracting the provided parameters into a dictionary and listing default values
        params_dict = eval(params) if type(params) == str else params
        X = {'level':50, 'critical':1, 'power':50, 'a':100, 'd':100, 'stab':1,
            'type':1, 'item':1, 'tk':1, 'weather':1, 'badge':1, 'movemod':1,
            'doubledmg':1, 'burn':1, 'screen':1, 'targets':1, 'ff':1, 'stockpile':1, 'charge':1,
            'hh':1, 'first':1, 'srf':1, 'eb':1, 'tl':1, 'berry':1, 'pb':1, 'glaiverush':1, 'other':1,
            'zmove':1, 'terashield':1, 'random':1, 'gen':9}

        # For each parameter, change the default to the specified value
        for key, val in params_dict.items():
            if key in X.keys():
                X[key.lower()] = float(val)
            else:
                return f'{key} is not a valid parameter, please see DamageCalcInfo'

        # --- Calculation ---
        if X['gen'] == 1:
            dmg = (((2*X['level']*X['critical']/5 + 2) * X['power'] * X['a'] / X['d']) / 50 + 2) * X['stab']*X['type']

        elif X['gen'] == 2:
            r = np.random.uniform(217, 255) / 255 if 'random' not in params_dict.keys() else X['random']
            dmg = ((((2*X['level']/5 + 2) * X['power'] * X['a'] / X['d']) / 50) * X['item']*X['critical'] + 2) * X['tk']*X['weather']*X['badge']*X['stab']*X['type']*X['movemod']*r*X['doubledmg']

        elif X['gen'] == 3:
            r = np.random.uniform(85, 100) / 100 if 'random' not in params_dict.keys() else X['random']
            dmg = ((((2*X['level']/5 + 2) * X['power'] * X['a'] / X['d']) / 50) * X['burn']*X['screen']*X['targets']*X['weather']*X['ff'] + 2) * X['stockpile']*X['critical']*X['doubledmg']*X['charge']*X['hh']*X['stab']*X['type']*r

        elif X['gen'] == 4:
            r = np.random.uniform(85, 100) / 100 if 'random' not in params_dict.keys() else X['random']
            dmg = ((((2*X['level']/5 + 2) * X['power'] * X['a'] / X['d']) / 50) * X['burn']*X['screen']*X['targets']*X['weather']*X['ff'] + 2) * X['critical']*X['item']*X['first']*X['stab']*X['type']*X['srf']*X['eb']*X['tl']*X['berry']*r

        else:
            r = np.random.uniform(85, 100) / 100 if 'random' not in params_dict.keys() else X['random']
            dmg = (((2*X['level']/5 + 2) * X['power'] * X['a'] / X['d']) / 50 + 2) * X['targets']*X['pb']*X['weather']*X['glaiverush']*X['critical']*r*X['stab']*X['type']*X['burn']*X['other']*X['zmove']*X['terashield']
        # --- ---

        # Rounding and final adjustment
        dmg = np.floor(dmg)
        if dmg == 0 and X['type'] != 0:
            dmg = 1

        return dmg
    except:
        return 'Incorrect format of provided dictionary, please see DamageCalcInfo'

def stat_calc(params:str) -> int:
    """
    This function can be used to calculate a stat in the Pokémon games with specified parameters.

    Args:
        params (str): All parameters, including generation, "gen", and HP, "hp" (either 1 to indicate HP stat or 0 otherwise), provided in a dictionary format as a string literal.

    Returns:
        int: The calculated stat.
    """
    try:
        # Extracting the provided parameters into a dictionary and listing default values
        params_dict = eval(params) if type(params) == str else params
        X = {'base':50, 'dv':1, 'statexp':65535, 'level':50, 'iv':31, 'ev':252, 'nature':1,
            'stat':"attack", 'gen':9}

        # For each parameter, change the default to the specified value
        for key, val in params_dict.items():
            if key in X.keys():
                X[key.lower()] = float(val) if key != 'stat' else val
            else:
                return f'{key} is not a valid parameter, please see StatCalcInfo'

        # If nature is accidentally passed as 110% etc, convert to 1.1 etc
        if X['nature'] > 10:
            X['nature'] /= 10

        # --- Calculation ---
        if X['gen'] <= 2:
            if X['stat'] == 'hp':
                stat = ((X['base'] + X['dv']) * 2 + np.sqrt(X['statexp']) / 4) * X['level'] / 100 + X['level'] + 10
            else:
                stat = ((X['base'] + X['dv']) * 2 + np.sqrt(X['statexp']) / 4) * X['level'] / 100 + 5

        elif X['gen'] >= 3:
            if X['stat'] == 'hp':
                stat = (2*X['base'] + X['iv'] + X['ev']/4) * X['level'] / 100 + X['level'] + 10
            else:
                stat = ((2*X['base'] + X['iv'] + X['ev']/4) * X['level'] / 100 + 5) * X['nature']
        # --- ---

        return stat
    except:
        return 'Incorrect format of provided dictionary, please see StatCalcInfo'


## 2. LLM Setup
Set up the LLM agent and include the tools defined above.

In [95]:
# Initialising the LLM
llm = ChatOpenAI(model='gpt-4.1-mini')

# Gathering all tools and providing comprehensive descriptions
tools = [
    Tool(name='PokemonInfoTypeEvos',
        description='Use this tool to retrieve some key facts about a Pokémon specified by name, including their Pokédex entries, type, and evolutions.',
        func=info_type_evos),
    Tool(name='PokemonBaseStats',
        description='Use this tool to retrieve the base stats of a Pokémon specified by name. These are used to calculate the effective stat using StatCalculator which is then used for damage calculations.',
        func=stats),
    Tool(name='PokemonMoves',
        description='Use this tool to retrieve a list of all moves learned by a Pokémon specified by name. To receieve the learnset for a specified generation, suffix the name with "|[generation number]"',
        func=learnset),
    Tool(name='MoveDetails',
        description='Use this tool to retrieve of a move specified by name, including additional effects.',
        func=move),
    Tool(name='Games',
        description='Use this tool to retrieve information about the main series games, which generations they all belong to.',
        func=games),
    Tool(name='TypeAdvantages',
        description='Use this tool to retrieve information about how effective one type is against another. Format the input as "[offensive type]|[defensive type]".',
        func=type_adv),
    Tool(name='DamageCalcInfo',
         description='Use this tool to retrieve information on the parameters used in calculating damage within a specified generation. Specify generation as "1" etc.',
         func=damage_calc_info),
    Tool(name='StatCalcInfo',
         description='Use this tool as retrieval augmented generation (RAG) to retrieve information on how stats are calculated within the games, including natures. Pass a query as input.',
         func=stat_calc_info),
    Tool(name='DamageCalculator',
         description="""Use this tool to calculate damage by passing specified parameters in a Python dictionary format embedded within a string literal. Keys for this dictionary
                        should match the parameters retrieved by DamageCalcInfo. All valid parameters with default values are '{'level':50, 'critical':1, 'power':50, 'a':100, 'd':100, 'stab':1,
                        'type':1, 'item':1, 'tk':1, 'weather':1, 'badge':1, 'movemod':1, 'doubledmg':1, 'burn':1, 'screen':1, 'targets':1, 'ff':1, 'stockpile':1, 'charge':1,
                        'hh':1, 'first':1, 'srf':1, 'eb':1, 'tl':1, 'berry':1, 'pb':1, 'glaiverush':1, 'other':1, 'zmove':1, 'terashield':1, 'random':1, 'gen':9}'. Attack and defense stats should
                        be specified using the keys "a" and "d" respectively (this is the same for both special and physical). Type advantage/disadvantage should be specified as a multiplier, 2
                        for super effective, 0.5 for not very effective, 1 for neutrally effective, and 0 for no effect. For dual typings, the advantage/disadvantage for for both types should be
                        considered and multiplied together.""",
         func=damage_calc),
    Tool(name='StatCalculator',
         description="""Use this tool to calculate an effective stat by passing specified parameters in a Python dictionary format embedded within a string literal. Keys for this dictionary
                        should match the parameters retrieved by StatCalcInfo. All valid parameters with default values are '{'base':50, 'dv':1, 'statexp':65535, 'level':50, 'iv':31, 'ev':252,
                        'nature':1, 'stat':"attack", 'gen':9}'. The stat parameter should be either "hp", "attack", "defense", "special attack", "special defense", or "speed".""",
         func=stat_calc)
]

# Pulling a tools agent prompt template
prompt = hub.pull("hwchase17/openai-tools-agent")

# Creating the agent executor
agent = create_openai_tools_agent(llm, tools, prompt)
agent_exec = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 3. Example Prompts

In [96]:
agent_exec.invoke({'input':'Would a thunderbolt from a typical level 50 Raichu knock out a typical level 45 Gyarados in Pokémon Scarlet?'})



> Entering new AgentExecutor chain...

Invoking: `PokemonBaseStats` with `Raichu`




HP
60

Min Max
230
324

Attack
90

Min Max
166
306

Defense
55

Min Max
103
229

Sp. Atk
90

Min Max
166
306

Sp. Def
80

Min Max
148
284

Speed
110

Min Max
202
350



Total
485





Invoking: `PokemonBaseStats` with `Gyarados`




HP
95

Min Max
300
394

Attack
125

Min Max
229
383

Defense
79

Min Max
146
282

Sp. Atk
60

Min Max
112
240

Sp. Def
100

Min Max
184
328

Speed
81

Min Max
150
287



Total
540





Invoking: `MoveDetails` with `Thunderbolt`



Type	Electric
Category	 Special
Power	90
Accuracy	100
PP	15 (max. 24)
Makes contact?	No
Introduced	Generation 1



Thunderbolt deals damage and has a 10% chance of paralyzing the target.
Electric type Pokémon, those with the ability Limber or those behind a Substitute cannot be paralyzed.


In Generations 1-5, Thunderbolt has 95 base power.

When a Pokémon is holding Electrium Z and uses its Z-Power, Thunderbolt turns into Gigavolt Havoc and ha

{'input': 'Would a thunderbolt from a typical level 50 Raichu knock out a typical level 45 Gyarados in Pokémon Scarlet?',
 'output': "A typical level 50 Raichu's Thunderbolt would deal approximately 229 HP damage to a typical level 45 Gyarados. Since the Gyarados has around 183 HP, Thunderbolt from Raichu would be enough to knock out the Gyarados in one hit in Pokémon Scarlet."}

In [97]:
agent_exec.invoke({'input':'What is Metang\'s weight?'})



> Entering new AgentExecutor chain...

Invoking: `PokemonInfoTypeEvos` with `Metang`


Metang is a Steel/Psychic type Pokémon introduced in <abbr title="Pokémon Ruby, Sapphire, FireRed, LeafGreen &amp; Emerald">Generation 3</abbr>.

National &#8470;
0375

Type

SteelPsychic

Species
Iron Claw Pokémon

Height
1.2&nbsp;m (3&#8242;11&#8243;)

Weight
202.5&nbsp;kg (446.4&nbsp;lbs)

Abilities
1. Clear Body Light Metal (hidden ability) 

Local &#8470;
0191 (Ruby/Sapphire/Emerald) 0263 (Black 2/White 2) 0200 (Omega Ruby/Alpha Sapphire) 0215 (Sun/Moon &mdash; Alola dex) 0279 (U.Sun/U.Moon &mdash; Alola dex) 0226 (Legends: Z-A) 0130 (The Crown Tundra) 0138 (The Indigo Disk) 



EV yield

2 Defense

Catch rate

3 (0.4% with PokéBall, full HP)


Base Friendship

35 (lower than normal)


Base Exp.
147

Growth Rate
Slow



Egg Groups
Mineral

Gender
Genderless

Egg cycles
40(10,024&ndash;10,280 steps)


#0374 Beldum Steel Psychic
(Level 20)#0375 Metang Steel Psychic
(Level 45)#0376 Metagross Stee

{'input': "What is Metang's weight?",
 'output': "Metang's weight is 202.5 kg (446.4 lbs)."}

In [98]:
agent_exec.invoke({'input':'How do I evolve my Feebas in Pokémon HeartGold?'})



> Entering new AgentExecutor chain...

Invoking: `PokemonInfoTypeEvos` with `Feebas`


Feebas is a Water type Pokémon introduced in <abbr title="Pokémon Ruby, Sapphire, FireRed, LeafGreen &amp; Emerald">Generation 3</abbr>.
Feebas is very rare in Ruby/Sapphire/Emerald, being available by fishing in one of only six random squares on Route 119.
Feebas also has a unique evolution.

National &#8470;
0349

Type

Water

Species
Fish Pokémon

Height
0.6&nbsp;m (2&#8242;00&#8243;)

Weight
7.4&nbsp;kg (16.3&nbsp;lbs)

Abilities
1. Swift Swim 2. Oblivious Adaptability (hidden ability) 

Local &#8470;
0140 (Ruby/Sapphire/Emerald) 0138 (Diamond/Pearl) 0138 (Platinum) 0145 (Omega Ruby/Alpha Sapphire) 0155 (Sun/Moon &mdash; Alola dex) 0189 (U.Sun/U.Moon &mdash; Alola dex) 0152 (Sword/Shield) 0138 (Brilliant Diamond/Shining Pearl) 0188 (The Crown Tundra) 0158 (The Teal Mask) 0048 (Mega Dimension) 



EV yield

1 Speed

Catch rate

255 (33.3% with PokéBall, full HP)


Base Friendship

50 (normal)




{'input': 'How do I evolve my Feebas in Pokémon HeartGold?',
 'output': "In Pokémon HeartGold, Feebas evolves into Milotic by leveling it up while it has the highest Beauty condition. You can increase Feebas's Beauty by using PokéBlocks, which are made from Berries in the Pokéblock Case. Once Feebas has the highest Beauty condition in your party, level it up, and it will evolve into Milotic."}

In [99]:
agent_exec.invoke({'input':'Can klefki learn moonblast in Pokémon Y?'})



> Entering new AgentExecutor chain...

Invoking: `PokemonMoves` with `klefki|6`


In X/Y:

Moves learnt by level up
Lv. Move Type Cat. Power Acc.
1 Fairy Lock Fairy Physical  50  100  
5 Fairy Wind Fairy Special  40  100  
8 Astonish Ghost Physical  30  100  
12 Metal Sound Steel Status  —  85  
15 Spikes Ground Status  —  —  
18 Draining Kiss Fairy Special  50  100  
23 Crafty Shield Fairy Status  —  —  
27 Foul Play Dark Physical  95  100  
32 Torment Dark Status  —  100  
34 Mirror Shot Steel Special  65  85  
36 Imprison Psychic Status  —  —  
40 Recycle Normal Status  —  —  
43 Play Rough Fairy Physical  90  90  
44 Magic Room Psychic Status  —  —  
50 Heal Block Psychic Status  —  100  

Egg moves
Move Type Cat. Power Acc.
Iron Defense Steel Status  —  —  
Switcheroo Dark Status  —  100  
Thief Dark Physical  60  100  

Moves learnt by TM
TM Move Type Cat. Power Acc.
03 Psyshock Psychic Status  —  —  
06 Toxic Poison Status  —  90  
10 Hidden Power Normal Special  60  100  
11 

{'input': 'Can klefki learn moonblast in Pokémon Y?',
 'output': 'In Pokémon Y, Klefki cannot learn the move Moonblast. It can learn other Fairy-type moves such as Fairy Lock, Fairy Wind, Draining Kiss, Play Rough, and Dazzling Gleam, but Moonblast is not among its learnable moves in this generation.'}